In [1]:
import shutil
import pandas as pd

shutil.copy("airbnb_scraped.csv", "airbnb_bronze_raw.csv")

bronze = pd.read_csv("airbnb_bronze_raw.csv")
print(bronze.shape)

(500, 11)


In [2]:
import pandas as pd

df = pd.read_csv("airbnb_bronze_raw.csv")
df.head()

,name,location,price_per_night_usd,rating,reviews_count,is_superhost,guest_favorite,bedrooms,url,city,country
0,HOUSEBOAT NOVA 80m2 + FREE BIKES,"Amsterdam, Netherlands",205.0,4.74,1249.0,False,False,2,https://www.airbnb.com/rooms/851044,Amsterdam,Netherlands
1,Bright Family Home in Amsterdam + Fast City Ac...,"Amsterdam, Netherlands",265.0,5.00,7.0,False,True,2,https://www.airbnb.com/rooms/1555751798744378815,Amsterdam,Netherlands
2,"2 room studio in city villa, close to centre.","Amsterdam, Netherlands",156.2,4.80,201.0,False,False,2,https://www.airbnb.com/rooms/699846140916875590,Amsterdam,Netherlands
3,Luxury Rijksmuseum House,"Amsterdam, Netherlands",553.6,4.98,174.0,True,True,2,https://www.airbnb.com/rooms/668118568598595862,Amsterdam,Netherlands
4,"Close to Center, Jordaan, Park, yet very quiet","Amsterdam, Netherlands",272.6,4.98,46.0,False,True,2,https://www.airbnb.com/rooms/13441385,Amsterdam,Netherlands


In [3]:
df = df[~df["name"].str.contains("Listing Editor", case=False, na=False)].reset_index(drop=True)
print(len(df))

499


In [4]:
print("rows:", len(df))
print("duplicate urls:", df.duplicated(subset="url").sum())

df["listing_id"] = df["url"].str.extract(r"/rooms/(\d+)")[0]
print("missing listing_id:", df["listing_id"].isna().sum())
print("duplicate listing_id:", df["listing_id"].duplicated().sum())

df[["listing_id", "city", "url"]].head(3)

rows: 499
duplicate urls: 0
missing listing_id: 0
duplicate listing_id: 0


,listing_id,city,url
0,851044,Amsterdam,https://www.airbnb.com/rooms/851044
1,1555751798744378815,Amsterdam,https://www.airbnb.com/rooms/1555751798744378815
2,699846140916875590,Amsterdam,https://www.airbnb.com/rooms/699846140916875590


In [5]:
df["has_reviews"] = df["reviews_count"].notna()
df["location"] = df["location"].fillna(df["city"] + ", " + df["country"])

print(df.isna().sum())
print("listings without reviews:", (~df["has_reviews"]).sum())

name                    0
location                0
price_per_night_usd     0
rating                 41
reviews_count          41
is_superhost            0
guest_favorite          0
bedrooms                0
url                     0
city                    0
country                 0
listing_id              0
has_reviews             0
dtype: int64
listings without reviews: 41


In [6]:
df["listing_id"] = df["listing_id"].astype("string")
df["reviews_count"] = df["reviews_count"].astype("Int64")
df["bedrooms"] = df["bedrooms"].astype("int64")
df["price_per_night_usd"] = df["price_per_night_usd"].astype("float64")
df["is_superhost"] = df["is_superhost"].astype(bool)
df["guest_favorite"] = df["guest_favorite"].astype(bool)

df = df[[
    "listing_id", "name", "city", "country", "location",
    "price_per_night_usd", "rating", "reviews_count", "has_reviews",
    "is_superhost", "guest_favorite", "bedrooms", "url",
]]

df.dtypes

listing_id              string
name                       str
city                       str
country                    str
location                   str
price_per_night_usd    float64
rating                 float64
reviews_count            Int64
has_reviews               bool
is_superhost              bool
guest_favorite            bool
bedrooms                 int64
url                        str
dtype: object

In [7]:
df.to_csv("airbnb_silver.csv", index=False, encoding="utf-8-sig")

check = pd.read_csv("airbnb_silver.csv")
print(check.shape)
print(check.isna().sum().sum(), "missing values in total")
check.head(3)

(499, 13)
82 missing values in total


,listing_id,name,city,country,location,price_per_night_usd,rating,reviews_count,has_reviews,is_superhost,guest_favorite,bedrooms,url
0,851044,HOUSEBOAT NOVA 80m2 + FREE BIKES,Amsterdam,Netherlands,"Amsterdam, Netherlands",205.0,4.74,1249.0,True,False,False,2,https://www.airbnb.com/rooms/851044
1,1555751798744378815,Bright Family Home in Amsterdam + Fast City Ac...,Amsterdam,Netherlands,"Amsterdam, Netherlands",265.0,5.00,7.0,True,False,True,2,https://www.airbnb.com/rooms/1555751798744378815
2,699846140916875590,"2 room studio in city villa, close to centre.",Amsterdam,Netherlands,"Amsterdam, Netherlands",156.2,4.80,201.0,True,False,False,2,https://www.airbnb.com/rooms/699846140916875590


In [8]:
import os

silver = pd.read_csv("airbnb_silver.csv", dtype={"listing_id": "string"})
silver["reviews_count"] = silver["reviews_count"].astype("Int64")

dim_city = silver[["city", "country"]].drop_duplicates().sort_values("city").reset_index(drop=True)
dim_city.insert(0, "city_key", dim_city.index + 1)

os.makedirs("gold", exist_ok=True)
dim_city.to_csv("gold/dim_city.csv", index=False, encoding="utf-8-sig")

dim_city

,city_key,city,country
0,1,Amsterdam,Netherlands
1,2,Athens,Greece
2,3,Barcelona,Spain
3,4,Berlin,Germany
4,5,Budapest,Hungary
5,6,Lisbon,Portugal
6,7,London,United Kingdom
7,8,Paris,France
8,9,Rome,Italy
9,10,Vienna,Austria


In [9]:
scrape_date = pd.Timestamp("2026-10-19")

dim_date = pd.DataFrame({
    "date_key": [int(scrape_date.strftime("%Y%m%d"))],
    "full_date": [scrape_date.date()],
    "year": [scrape_date.year],
    "month": [scrape_date.month],
    "month_name": [scrape_date.strftime("%B")],
    "day": [scrape_date.day],
})

dim_date.to_csv("gold/dim_date.csv", index=False, encoding="utf-8-sig")

dim_date

,date_key,full_date,year,month,month_name,day
0,20261019,2026-10-19,2026,10,October,19


In [10]:
dim_listing = silver[["listing_id", "name", "url", "bedrooms", "location", "is_superhost", "guest_favorite"]]
dim_listing = dim_listing.drop_duplicates(subset="listing_id").reset_index(drop=True)
dim_listing.insert(0, "listing_key", dim_listing.index + 1)

dim_listing.to_csv("gold/dim_listing.csv", index=False, encoding="utf-8-sig")

print("rows:", len(dim_listing))
print("duplicate listing_id:", dim_listing["listing_id"].duplicated().sum())

dim_listing.head()

rows: 499
duplicate listing_id: 0


,listing_key,listing_id,name,url,bedrooms,location,is_superhost,guest_favorite
0,1,851044,HOUSEBOAT NOVA 80m2 + FREE BIKES,https://www.airbnb.com/rooms/851044,2,"Amsterdam, Netherlands",False,False
1,2,1555751798744378815,Bright Family Home in Amsterdam + Fast City Ac...,https://www.airbnb.com/rooms/1555751798744378815,2,"Amsterdam, Netherlands",False,True
2,3,699846140916875590,"2 room studio in city villa, close to centre.",https://www.airbnb.com/rooms/699846140916875590,2,"Amsterdam, Netherlands",False,False
3,4,668118568598595862,Luxury Rijksmuseum House,https://www.airbnb.com/rooms/668118568598595862,2,"Amsterdam, Netherlands",True,True
4,5,13441385,"Close to Center, Jordaan, Park, yet very quiet",https://www.airbnb.com/rooms/13441385,2,"Amsterdam, Netherlands",False,True


In [11]:
fact = silver.merge(dim_city, on=["city", "country"]).merge(dim_listing[["listing_key", "listing_id"]], on="listing_id")
fact["date_key"] = dim_date.loc[0, "date_key"]

fact_listing_snapshot = fact[[
    "listing_key", "city_key", "date_key",
    "price_per_night_usd", "rating", "reviews_count", "has_reviews",
]].sort_values("listing_key").reset_index(drop=True)

fact_listing_snapshot.to_csv("gold/fact_listing_snapshot.csv", index=False, encoding="utf-8-sig")

print("rows:", len(fact_listing_snapshot))
print("null keys:", fact_listing_snapshot[["listing_key", "city_key", "date_key"]].isna().sum().sum())
print("duplicate listing_key:", fact_listing_snapshot["listing_key"].duplicated().sum())
print("orphan city_key:", (~fact_listing_snapshot["city_key"].isin(dim_city["city_key"])).sum())
print("orphan listing_key:", (~fact_listing_snapshot["listing_key"].isin(dim_listing["listing_key"])).sum())

fact_listing_snapshot.head()

rows: 499
null keys: 0
duplicate listing_key: 0
orphan city_key: 0
orphan listing_key: 0


,listing_key,city_key,date_key,price_per_night_usd,rating,reviews_count,has_reviews
0,1,1,20261019,205.0,4.74,1249,True
1,2,1,20261019,265.0,5.00,7,True
2,3,1,20261019,156.2,4.80,201,True
3,4,1,20261019,553.6,4.98,174,True
4,5,1,20261019,272.6,4.98,46,True


In [12]:
city_summary = (
    fact_listing_snapshot
    .merge(dim_city, on="city_key")
    .merge(dim_listing[["listing_key", "is_superhost", "guest_favorite"]], on="listing_key")
    .groupby(["city_key", "city", "country"], as_index=False)
    .agg(
        listings=("listing_key", "count"),
        avg_price=("price_per_night_usd", "mean"),
        median_price=("price_per_night_usd", "median"),
        avg_rating=("rating", "mean"),
        superhost_pct=("is_superhost", "mean"),
        guest_favorite_pct=("guest_favorite", "mean"),
        new_listings=("has_reviews", lambda s: (~s).sum()),
    )
)

city_summary["superhost_pct"] = (city_summary["superhost_pct"] * 100).round(1)
city_summary["guest_favorite_pct"] = (city_summary["guest_favorite_pct"] * 100).round(1)
city_summary[["avg_price", "median_price", "avg_rating"]] = city_summary[["avg_price", "median_price", "avg_rating"]].round(2)

city_summary.to_csv("gold/city_summary.csv", index=False, encoding="utf-8-sig")

city_summary.sort_values("avg_price", ascending=False)

,city_key,city,country,listings,avg_price,median_price,avg_rating,superhost_pct,guest_favorite_pct,new_listings
0,1,Amsterdam,Netherlands,50,306.73,304.30,4.86,68.0,56.0,2
7,8,Paris,France,50,294.23,249.05,4.89,56.0,54.0,9
6,7,London,United Kingdom,50,266.45,256.20,4.86,42.0,40.0,11
3,4,Berlin,Germany,50,185.01,179.80,4.91,84.0,72.0,5
2,3,Barcelona,Spain,50,184.83,165.30,4.80,68.0,42.0,1
9,10,Vienna,Austria,50,180.65,167.70,4.89,64.0,58.0,3
8,9,Rome,Italy,49,178.17,168.40,4.91,61.2,81.6,1
5,6,Lisbon,Portugal,50,152.52,135.30,4.87,66.0,66.0,2
4,5,Budapest,Hungary,50,107.84,94.70,4.89,64.0,64.0,6
1,2,Athens,Greece,50,89.44,80.20,4.90,64.0,68.0,1


In [13]:
silver[(silver["city"] == "Berlin") & (silver["is_superhost"])][["name", "url"]].head(5)


,name,url
150,"Cozy Apartment - Parking, Coffee, Netflix & more",https://www.airbnb.com/rooms/878313640342175143
151,Central flat near Alex with fiber optic Internet,https://www.airbnb.com/rooms/13760375
152,Spacious Parkside Residence * Modern 2BR by OYA,https://www.airbnb.com/rooms/1690465067267941577
153,2 Bedroom Apartment Large with Balcony,https://www.airbnb.com/rooms/31970794
154,Beautiful NEW 3 room in hip Prenzlauer Berg,https://www.airbnb.com/rooms/1350792206249789489


In [14]:
import sqlite3

conn = sqlite3.connect("airbnb_dwh.db")
conn.execute("PRAGMA foreign_keys = ON")

conn.executescript("""
DROP VIEW IF EXISTS vw_city_summary;
DROP TABLE IF EXISTS fact_listing_snapshot;
DROP TABLE IF EXISTS dim_listing;
DROP TABLE IF EXISTS dim_date;
DROP TABLE IF EXISTS dim_city;

CREATE TABLE dim_city (
    city_key INTEGER PRIMARY KEY,
    city TEXT NOT NULL,
    country TEXT NOT NULL
);

CREATE TABLE dim_date (
    date_key INTEGER PRIMARY KEY,
    full_date TEXT NOT NULL,
    year INTEGER NOT NULL,
    month INTEGER NOT NULL,
    month_name TEXT NOT NULL,
    day INTEGER NOT NULL
);

CREATE TABLE dim_listing (
    listing_key INTEGER PRIMARY KEY,
    listing_id TEXT NOT NULL UNIQUE,
    name TEXT,
    url TEXT,
    bedrooms INTEGER,
    location TEXT,
    is_superhost INTEGER,
    guest_favorite INTEGER
);

CREATE TABLE fact_listing_snapshot (
    listing_key INTEGER NOT NULL REFERENCES dim_listing(listing_key),
    city_key INTEGER NOT NULL REFERENCES dim_city(city_key),
    date_key INTEGER NOT NULL REFERENCES dim_date(date_key),
    price_per_night_usd REAL,
    rating REAL,
    reviews_count INTEGER,
    has_reviews INTEGER,
    PRIMARY KEY (listing_key, date_key)
);

CREATE VIEW vw_city_summary AS
SELECT
    c.city_key,
    c.city,
    c.country,
    COUNT(*) AS listings,
    ROUND(AVG(f.price_per_night_usd), 2) AS avg_price,
    ROUND(AVG(f.rating), 2) AS avg_rating,
    ROUND(100.0 * AVG(l.is_superhost), 1) AS superhost_pct,
    ROUND(100.0 * AVG(l.guest_favorite), 1) AS guest_favorite_pct,
    SUM(CASE WHEN f.has_reviews = 0 THEN 1 ELSE 0 END) AS new_listings
FROM fact_listing_snapshot f
JOIN dim_city c ON c.city_key = f.city_key
JOIN dim_listing l ON l.listing_key = f.listing_key
GROUP BY c.city_key, c.city, c.country;
""")

pd.read_csv("gold/dim_city.csv").to_sql("dim_city", conn, if_exists="append", index=False)
pd.read_csv("gold/dim_date.csv").to_sql("dim_date", conn, if_exists="append", index=False)
pd.read_csv("gold/dim_listing.csv", dtype={"listing_id": "string"}).to_sql("dim_listing", conn, if_exists="append", index=False)
pd.read_csv("gold/fact_listing_snapshot.csv").to_sql("fact_listing_snapshot", conn, if_exists="append", index=False)
conn.commit()

for t in ["dim_city", "dim_date", "dim_listing", "fact_listing_snapshot"]:
    print(t, conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0])

pd.read_sql("SELECT * FROM vw_city_summary ORDER BY avg_price DESC", conn)

dim_city 10
dim_date 1
dim_listing 499
fact_listing_snapshot 499


,city_key,city,country,listings,avg_price,avg_rating,superhost_pct,guest_favorite_pct,new_listings
0,1,Amsterdam,Netherlands,50,306.73,4.86,68.0,56.0,2
1,8,Paris,France,50,294.23,4.89,56.0,54.0,9
2,7,London,United Kingdom,50,266.45,4.86,42.0,40.0,11
3,4,Berlin,Germany,50,185.01,4.91,84.0,72.0,5
4,3,Barcelona,Spain,50,184.83,4.80,68.0,42.0,1
5,10,Vienna,Austria,50,180.65,4.89,64.0,58.0,3
6,9,Rome,Italy,49,178.17,4.91,61.2,81.6,1
7,6,Lisbon,Portugal,50,152.52,4.87,66.0,66.0,2
8,5,Budapest,Hungary,50,107.84,4.89,64.0,64.0,6
9,2,Athens,Greece,50,89.44,4.90,64.0,68.0,1
